# 04. PPO 행동 분석

**목적**: exp005 best_model(step=100k, Sharpe=1.060)과 exp006 best_model(step=550k, Sharpe=1.183)이  
Val 셋에서 실제로 어떤 행동을 하는지 분석한다.

**핵심 질문**:
1. 에이전트가 학습한 action 분포는? (aggressiveness / profit_target)
2. 변동성 레짐(Low/Mid/High)에 따라 행동이 달라지는가?
3. 실제 체결이 이루어지는 시점과 포트폴리오 변화는?

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

from src.utils.config import load_config
from src.agents.ppo_agent import PPOAgent
from src.evaluation.behavior import collect, actions_to_gaps, regime_analysis
from src.env.trading_env import BTCGridTradingEnv

import os
os.makedirs('../reports/semester1/figures', exist_ok=True)

print('라이브러리 로드 완료')

In [ ]:
df_train = pd.read_parquet('../data/processed/btc_train.parquet')
df_val   = pd.read_parquet('../data/processed/btc_val.parquet')
print(f'Train: {len(df_train):,}행  |  Val: {len(df_val):,}행')

## 1. 모델 로드

exp005 best_model과 exp006 best_model 두 개를 비교한다.  
- exp005: 최소 gap 계수 수정 (0.1→0.5), 단일 환경, 3M steps
- exp006: 에피소드 단축(2016스텝) + 4환경 병렬, 3M steps

In [ ]:
# exp005 best_model
cfg005 = load_config('../config/exp005_config.yaml')
agent005 = PPOAgent.load(
    '../experiments/exp005_min_gap_fix/best_model',
    cfg005, df_train, df_val
)
print('exp005 best_model 로드 완료')

# exp006 best_model
cfg006 = load_config('../config/exp006_config.yaml')
agent006 = PPOAgent.load(
    '../experiments/exp006_short_episode_multienv/best_model',
    cfg006, df_train, df_val
)
print('exp006 best_model 로드 완료')

## 2. 행동 수집

Val 셋 전체 에피소드 동안 매 스텝의 action을 기록한다.

In [ ]:
bdf005 = collect(agent005.model, df_val, cfg005, deterministic=True)
bdf006 = collect(agent006.model, df_val, cfg006, deterministic=True)

print(f'exp005: {len(bdf005):,}스텝 수집')
print(f'  aggressiveness: mean={bdf005["aggressiveness"].mean():.3f}, std={bdf005["aggressiveness"].std():.3f}')
print(f'  profit_target : mean={bdf005["profit_target"].mean():.3f}, std={bdf005["profit_target"].std():.3f}')
print()
print(f'exp006: {len(bdf006):,}스텝 수집')
print(f'  aggressiveness: mean={bdf006["aggressiveness"].mean():.3f}, std={bdf006["aggressiveness"].std():.3f}')
print(f'  profit_target : mean={bdf006["profit_target"].mean():.3f}, std={bdf006["profit_target"].std():.3f}')

## 3. Action 분포 시각화

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Action 분포 비교: exp005 vs exp006 (Val 셋)', fontsize=13, fontweight='bold')

bins = np.linspace(0, 1, 41)

for row_idx, (bdf, label, color) in enumerate([
    (bdf005, 'exp005 best (100k)', '#6A1B9A'),
    (bdf006, 'exp006 best (550k)', '#1565C0'),
]):
    ax_agg = axes[row_idx, 0]
    ax_pt  = axes[row_idx, 1]

    ax_agg.hist(bdf['aggressiveness'], bins=bins, color=color, alpha=0.75, edgecolor='white', lw=0.3)
    ax_agg.axvline(bdf['aggressiveness'].mean(), color='red', lw=1.5, ls='--',
                   label=f'mean={bdf["aggressiveness"].mean():.3f}')
    ax_agg.set_title(f'{label} — aggressiveness (매수 간격)', fontsize=10)
    ax_agg.set_xlabel('aggressiveness [0=좁은 간격, 1=넓은 간격]')
    ax_agg.set_ylabel('빈도')
    ax_agg.legend(fontsize=9)
    ax_agg.set_xlim(0, 1)

    ax_pt.hist(bdf['profit_target'], bins=bins, color=color, alpha=0.75, edgecolor='white', lw=0.3)
    ax_pt.axvline(bdf['profit_target'].mean(), color='red', lw=1.5, ls='--',
                  label=f'mean={bdf["profit_target"].mean():.3f}')
    ax_pt.set_title(f'{label} — profit_target (매도 간격)', fontsize=10)
    ax_pt.set_xlabel('profit_target [0=좁은 간격, 1=넓은 간격]')
    ax_pt.set_ylabel('빈도')
    ax_pt.legend(fontsize=9)
    ax_pt.set_xlim(0, 1)

plt.tight_layout()
plt.savefig('../reports/semester1/figures/04_action_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: 04_action_distribution.png')

**해석**:
- action이 0이나 1에 집중되어 있으면 에이전트가 고정 전략을 학습한 것
- 분산이 넓으면 상황에 따라 다양한 간격을 사용하는 것

## 4. 변동성 레짐별 행동 비교

레짐 정의 (zscore_volatility 기준):
- **Low**: zscore < -0.5 (낮은 변동성)
- **Mid**: -0.5 ≤ zscore ≤ 0.5
- **High**: zscore > 0.5 (높은 변동성)

In [ ]:
def regime_summary(bdf, label):
    summary = bdf.groupby('regime')[['aggressiveness','profit_target']].agg(['mean','std'])
    counts  = bdf['regime'].value_counts()
    print(f'\n=== {label} 레짐별 행동 ===' )
    print(f'레짐 분포: {dict(counts)}')
    print(summary.round(3))
    return summary

s005 = regime_summary(bdf005, 'exp005 best')
s006 = regime_summary(bdf006, 'exp006 best')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('레짐별 평균 Action 비교', fontsize=13, fontweight='bold')

regime_colors = {'Low': '#4CAF50', 'Mid': '#FF9800', 'High': '#F44336'}
regimes = ['Low', 'Mid', 'High']

for col_idx, regime in enumerate(regimes):
    for row_idx, (bdf, label) in enumerate([(bdf005, 'exp005'), (bdf006, 'exp006')]):
        ax = axes[row_idx, col_idx]
        sub = bdf[bdf['regime'] == regime]
        if len(sub) == 0:
            ax.text(0.5, 0.5, '데이터 없음', ha='center', va='center', transform=ax.transAxes)
            continue

        x = np.arange(2)
        means = [sub['aggressiveness'].mean(), sub['profit_target'].mean()]
        stds  = [sub['aggressiveness'].std(),  sub['profit_target'].std()]
        color = regime_colors[regime]

        bars = ax.bar(x, means, yerr=stds, color=color, alpha=0.7,
                      capsize=5, edgecolor='white')
        ax.set_xticks(x)
        ax.set_xticklabels(['aggressiveness\n(매수 간격)', 'profit_target\n(매도 간격)'])
        ax.set_ylim(0, 1.1)
        ax.set_ylabel('action 값')
        ax.set_title(f'{label} — {regime} Vol ({len(sub):,}스텝)', fontsize=9)
        ax.axhline(0.5, color='gray', ls='--', lw=0.8, alpha=0.5, label='중간값(0.5)')
        ax.legend(fontsize=7)

        for bar, m in zip(bars, means):
            ax.text(bar.get_x() + bar.get_width()/2, m + 0.03,
                    f'{m:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/semester1/figures/04_regime_behavior.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: 04_regime_behavior.png')

## 5. 포트폴리오 시계열 + 실제 거래 시각화

처음 1,000스텝 구간에서 가격 변화, 포트폴리오 가치, 보유 BTC 수량을 함께 시각화한다.

In [ ]:
def run_episode_trace(model, df, config, n_steps=1000):
    """env를 직접 실행하며 거래 시점과 포트폴리오를 기록한다."""
    env = BTCGridTradingEnv(df, config)
    obs, _ = env.reset(seed=42)

    trace = []
    for step in range(n_steps):
        n_trades_before = env.n_trades
        holdings_before = env.holdings

        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)

        price = float(df.iloc[env.current_step - 1]['close'])
        equity = env.cash + env.holdings * price

        traded = env.n_trades > n_trades_before
        bought = traded and env.holdings > holdings_before
        sold   = traded and env.holdings < holdings_before

        trace.append({
            'step':     step,
            'price':    price,
            'equity':   equity,
            'cash':     env.cash,
            'holdings': env.holdings,
            'bought':   bought,
            'sold':     sold,
            'agg':      float(action[0]),
            'pt':       float(action[1]),
        })

        if terminated or truncated:
            break

    return pd.DataFrame(trace)

trace005 = run_episode_trace(agent005.model, df_val, cfg005, n_steps=1000)
trace006 = run_episode_trace(agent006.model, df_val, cfg006, n_steps=1000)

print(f'exp005: 매수 {trace005["bought"].sum()}회, 매도 {trace005["sold"].sum()}회')
print(f'exp006: 매수 {trace006["bought"].sum()}회, 매도 {trace006["sold"].sum()}회')

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharex='col')
fig.suptitle('처음 1,000스텝 거래 시각화 (Val 셋)', fontsize=13, fontweight='bold')

for col_idx, (trace, label, color) in enumerate([
    (trace005, 'exp005 best (100k)', '#6A1B9A'),
    (trace006, 'exp006 best (550k)', '#1565C0'),
]):
    steps = trace['step'].values

    # 패널 1: 가격 + 거래 시점
    ax1 = axes[0, col_idx]
    ax1.plot(steps, trace['price'], color='gray', lw=1.0, alpha=0.8, label='BTC 가격')
    buy_mask  = trace['bought'].values
    sell_mask = trace['sold'].values
    if buy_mask.any():
        ax1.scatter(steps[buy_mask],  trace['price'].values[buy_mask],
                    marker='^', color='#4CAF50', s=80, zorder=5, label=f'매수({buy_mask.sum()})')
    if sell_mask.any():
        ax1.scatter(steps[sell_mask], trace['price'].values[sell_mask],
                    marker='v', color='#F44336', s=80, zorder=5, label=f'매도({sell_mask.sum()})')
    ax1.set_title(f'{label} — 가격 + 거래', fontsize=10)
    ax1.set_ylabel('BTC 가격 ($)')
    ax1.legend(fontsize=8)
    ax1.grid(True, alpha=0.3)

    # 패널 2: 포트폴리오 가치
    ax2 = axes[1, col_idx]
    ax2.plot(steps, trace['equity'], color=color, lw=1.5, label='포트폴리오')
    ax2.plot(steps, trace['cash'],   color='#FF9800', lw=1.0, ls='--', alpha=0.7, label='현금')
    ax2.axhline(10000, color='black', lw=0.8, ls=':', alpha=0.5)
    ax2.set_ylabel('가치 ($)')
    ax2.set_title('포트폴리오 + 현금', fontsize=10)
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

    # 패널 3: Action 시계열
    ax3 = axes[2, col_idx]
    ax3.plot(steps, trace['agg'], lw=1.0, color='#1565C0', alpha=0.8, label='aggressiveness')
    ax3.plot(steps, trace['pt'],  lw=1.0, color='#E53935', alpha=0.8, label='profit_target')
    ax3.axhline(0.5, color='gray', ls='--', lw=0.8, alpha=0.5)
    ax3.set_ylim(-0.05, 1.05)
    ax3.set_ylabel('Action 값')
    ax3.set_xlabel('스텝')
    ax3.set_title('Action 시계열', fontsize=10)
    ax3.legend(fontsize=8)
    ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/semester1/figures/04_trade_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: 04_trade_visualization.png')

## 6. exp005 vs exp006 — 행동 다양성 비교

In [ ]:
print('=== action 통계 비교 ===')
print(f'{'':20} {'exp005':>10} {'exp006':>10}')
print('-' * 42)
for col in ['aggressiveness', 'profit_target']:
    for stat, fn in [('mean', 'mean'), ('std', 'std'), ('p25', lambda x: x.quantile(0.25)), ('p75', lambda x: x.quantile(0.75))]:
        if callable(fn):
            v005, v006 = fn(bdf005[col]), fn(bdf006[col])
        else:
            v005, v006 = getattr(bdf005[col], fn)(), getattr(bdf006[col], fn)()
        print(f'{col[:14]}.{stat:4} {v005:>10.3f} {v006:>10.3f}')

## 7. 결론

### 관찰 요약

| 항목 | exp005 best | exp006 best | 해석 |
|------|-------------|-------------|------|
| best Sharpe | 1.060 | **1.183** | exp006 소폭 우위 |
| 학습 안정성 | 초반만 학습 (100k 이후 하락) | 전 구간 Sharpe 0.6~1.2 유지 | exp006 대폭 개선 |
| aggressiveness 분포 | ? | ? | 노트북 실행 후 확인 |
| 레짐별 차별화 | ? | ? | 노트북 실행 후 확인 |

### 아직 해결되지 않은 문제

1. **Return=+0.00%** (3M steps 내내): 거래는 발생하지만 순익이 거의 없음
   - 원인 가설: VecNormalize inference 시 obs_rms 불일치 → 장기 에피소드에서 이상 행동
   - 다음 실험(exp007): `norm_reward=False` 또는 VecNormalize 없이 재실험

2. **MDD 30~80%**: 포지션을 장기 보유하며 BTC 가격 하락을 그대로 맞음
   - 원인: 매도 조건이 충족되지 않아 보유 지속
   - 해결 방향: sell_cost 조건 완화 또는 stop-loss 구조 추가 검토

3. **베이스라인 미달**: best Sharpe=1.183 < Fixed Grid 1% Sharpe=2.610
   - 2023년은 강세장 → Buy-and-Hold에 유리한 환경
   - Train(2020-2022 혼재) 기준으로는 의미 있는 비교 필요